# Build AntiDown APK tren Google Colab

Notebook nay clone repo GitHub, cai Buildozer + Android toolchain, build APK debug, roi tai file APK ve may.

Cach dung:
1. Mo notebook nay tren Google Colab.
2. Chon `Runtime > Run all`.
3. Cho build xong, APK nam trong thu muc `bin/` va se duoc tu dong download.

Lan build dau co the mat 20-45 phut vi Colab phai tai SDK/NDK va bien dich dependency Android. Mac dinh notebook chi build `arm64-v8a` de tranh build 3 ABI qua lau.

In [ ]:
# Cau hinh repo
REPO_URL = "https://github.com/locntssj/AntiDown.git"
BRANCH = "main"
PROJECT_DIR = "/content/AntiDown"
TARGET_ABIS = ["arm64-v8a"]

# Bat True neu muon xoa source cu va clone lai tu dau.
CLEAN_SOURCE = True

# Bat True neu muon dung Google Drive de cache .buildozer cho lan build sau nhanh hon.
USE_DRIVE_CACHE = False

## 1. Cai goi he thong

In [ ]:
%%bash
set -euxo pipefail
sudo apt-get update
sudo apt-get install -y \
  git zip unzip openjdk-17-jdk python3-pip python3-setuptools python3-venv \
  build-essential ccache autoconf automake libtool pkg-config cmake \
  zlib1g-dev libffi-dev libssl-dev libncurses5-dev libncursesw5-dev

## 2. Cai Buildozer

In [ ]:
%%bash
set -euxo pipefail
python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install "cython<3.0" buildozer

## 3. Clone source

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if CLEAN_SOURCE and Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

if not Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=True)

os.chdir(PROJECT_DIR)
print("Project:", Path.cwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 4. Kiem tra file ffmpeg bundled

In [ ]:
from pathlib import Path
import os

missing = []
for abi in TARGET_ABIS:
    for name in ["ffmpeg", "ffprobe"]:
        path = Path("bin/android") / abi / name
        if not path.exists():
            missing.append(str(path))
        else:
            path.chmod(path.stat().st_mode | 0o755)
            bin_path = path.with_name(name + ".bin")
            if not bin_path.exists():
                bin_path.write_bytes(path.read_bytes())
            bin_path.chmod(bin_path.stat().st_mode | 0o755)
            native_dir = path.parent / "native"
            native_dir.mkdir(parents=True, exist_ok=True)
            native_name = "libantidown_ffmpeg.so" if name == "ffmpeg" else "libantidown_ffprobe.so"
            native_path = native_dir / native_name
            if not native_path.exists():
                native_path.write_bytes(path.read_bytes())
            native_path.chmod(native_path.stat().st_mode | 0o755)

if missing:
    raise FileNotFoundError("Thieu ffmpeg bundled: " + ", ".join(missing))

print("OK: ffmpeg/ffprobe Android da co san cho:", ", ".join(TARGET_ABIS))

## 4b. Toi uu Buildozer cho Colab

In [ ]:
from pathlib import Path

spec_path = Path(PROJECT_DIR) / "buildozer.spec"
spec_text = spec_path.read_text(encoding="utf-8")
abi_text = ", ".join(TARGET_ABIS)
include_patterns = ",".join(
    f"bin/android/{abi}/*,bin/android/{abi}/lib/*" for abi in TARGET_ABIS
)

lines = []
for line in spec_text.splitlines():
    if line.startswith("android.archs ="):
        lines.append(f"android.archs = {abi_text}")
    elif line.startswith("source.include_patterns ="):
        lines.append(f"source.include_patterns = {include_patterns}")
    elif line.startswith("warn_on_root ="):
        lines.append("warn_on_root = 0")
    else:
        lines.append(line)

spec_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Build ABIs:", abi_text)
print("Updated", spec_path)

## 5. Tuy chon cache Buildozer bang Google Drive

Co the de `USE_DRIVE_CACHE = False` cho lan build dau. Neu build nhieu lan, doi thanh `True` o cell cau hinh de Colab luu SDK/NDK vao Drive.

In [ ]:
from pathlib import Path
import os
import shutil

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache_dir = Path("/content/drive/MyDrive/AntiDownBuildozerCache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    local_buildozer = Path(PROJECT_DIR) / ".buildozer"
    if local_buildozer.exists() and not local_buildozer.is_symlink():
        shutil.rmtree(local_buildozer)
    if not local_buildozer.exists():
        local_buildozer.symlink_to(cache_dir, target_is_directory=True)
    print("Buildozer cache:", cache_dir)
else:
    print("Khong dung Drive cache cho lan build nay.")

## 6. Build APK debug

In [ ]:
%%bash
set -euxo pipefail
cd /content/AntiDown
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH="$JAVA_HOME/bin:$PATH"
export CI=1
stdbuf -oL -eL buildozer -v android debug 2>&1 | tee /content/antidown-build.log

## 7. Tai APK ve may

In [ ]:
from pathlib import Path
from google.colab import files

apk_files = sorted(Path(PROJECT_DIR, "bin").glob("*.apk"), key=lambda p: p.stat().st_mtime, reverse=True)
if not apk_files:
    raise FileNotFoundError("Khong tim thay APK trong /content/AntiDown/bin")

apk = apk_files[0]
print("APK:", apk, "size:", round(apk.stat().st_size / 1024 / 1024, 2), "MB")
files.download(str(apk))

## Lenh cai len Android bang ADB neu can

Sau khi tai APK ve may Windows, ket noi dien thoai bat USB debugging roi chay:

```powershell
adb install -r path\to\AntiDown-0.1.0-arm64-v8a-debug.apk
```

Ten file APK co the khac tuy Buildozer.